# **HydroServer Exercise 2 - Kenya: Loading Real-time Sensor Data into HydroServer**

### **Overview**

In this exercise, we will use the **Kanzenze Hydrological Station in Rwanda** as an example to demonstrate how real-time telemetry data can be loaded into HydroServer. Kanzenze is an existing hydrological monitoring station that was upgraded and equipped with a modern telemetry system through the **Nile Basin Initiative (NBI) HydroMet project**.

The station provides several hydrological time series through the **Rwanda Water Portal**, including both historical and telemetry observations. For this exercise, we will use the **Stage – Telemetry2** time series, which provides river stage observations from **September 2022 to the present**. The data can be accessed directly through the [Rwanda Water Portal](https://waterportal.rwb.rw/index.php/location_ng_info/259501).

Following this example, you will create a monitoring site and a datastream in the workspace you set up in Exercise 1. Later, you will use this datastream to load observations using the [HydroServer Streaming Data Loader](https://hydroserver.org/user-guides/tutorials/hydroserver-101/part-3-sdl-setup.html).



## 1. **Getting Started**

---

### **Install hydroserverpy**

For this workshop, we will use Google Colab to run the exercises. Before starting, run the code cell below to install the required version of the hydroserverpy package. For this exercise, we will connect to the HydroServer Playground instance at playground.hydroserver.org. The current version of hydroserverpy used for this training is [1.11.3.](https://pypi.org/project/hydroserverpy/)

In [ ]:
!pip install hydroserverpy==1.11.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.3 MB/s eta 0:00:00


### **Import Required Packages**

The following packages and modules are used in this exercise:

- **hydroserverpy** – Connects to HydroServer and allows us to create and manage HydroServer resources programmatically.
- **pandas** – Reads, organizes, and processes historical sensor data.
- **datetime** – Works with dates and times.
- **getpass** – Allows you to enter your HydroServer password securely without displaying it on the screen.

In [ ]:
# Import HydroServer to connect to and manage HydroServer resources
from hydroserverpy import HydroServer
# Import pandas to read, organize, and process the sensor data
import pandas as pd
# Import datetime to work with dates and times
from datetime import datetime
# Import getpass to securely enter your HydroServer password
from getpass import getpass

### **Set the Initial Parameters to Connect to HydroServer**

The first step in interacting with a HydroServer instance is to create a connection to that instance. For this example, we will use your username and password because we will create the Workspace in code.

**IMPORTANT: In the following code, change the email to match the HydroServer user account you created.**

In [ ]:
# Set initial parameters to connect to HydroServer
hydroserver_host = 'https://playground.hydroserver.org'

# Change the email and password below to your HydroServer username and password
hydroserver_email = 'svicario@lincolninst.edu' #'user@youremail.com'
hydroserver_password = getpass('Enter your HydroServer password: ') #getpass('Enter your HydroServer password: ')

Enter your HydroServer password: ··········


### **Initialize HydroServer Connection**

Initialize the connection to HydroServer with the connection information specified above.

In [ ]:
# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')


Successfully connected to HydroServer!


### **Get the ID of your workspace**

In [ ]:
# Enter the name of the workspace you created in Exercise 1
workspace_name = "Kenya Training 2026 - Sara"

workspaces = hs.workspaces.list(
    is_associated=True
).fetch_all()

workspace_id = next(
    workspace.uid
    for workspace in workspaces.items
    if workspace.name == workspace_name
)

print("Selected workspace:", workspace_name)
print("Workspace ID:", workspace_id)

StopIteration: 

## 2. **Create the Monitoring Site**

We are going to load historical river stage data from the Kanzenze Hydrological Station, located on the Akagera River between the Kicukiro and Bugesera districts in Rwanda.

First, we need to create a monitoring site, also referred to as a Thing in HydroServer. HydroServer uses a modified version of the OGC SensorThings API data model, where a Thing represents a monitoring location where observations are collected.

Once the monitoring site is created, HydroServer automatically assigns it a Universally Unique Identifier (UUID). We can use this UUID to build the URL for the site's landing page in HydroServer.

In [ ]:
# Create a Thing for the Kanzenze Hydrological Station

new_thing = hs.things.create(
    workspace=workspace_id,
    name='Kanzenze Hydrological Station',
    description='Hydrological monitoring station on the Nyabarongo River at Kanzenze, Rwanda.',
    sampling_feature_type='Site',
    sampling_feature_code='259501',
    site_type='Stream',
    elevation_m=1338.0,
    latitude=-2.0613,
    longitude=30.0877,
    admin_area_1='Eastern Province',
    admin_area_2='Bugesera',
    country='RW',
    data_disclaimer='Data provided by the Rwanda Water Resources Board (RWB).',
    is_private=False
)

# Get the ID for the new Thing and print its HydroServer landing page

thing_id = new_thing.uid

print(f'Created new thing with ID: {thing_id}')
print('You can access the new Thing in the HydroServer Data Management App at:')
print(f'{hydroserver_host}/sites/{thing_id}')

## 3. **Create the Datastream**

---

In the following sections, we will create the necessary metadata to load data for a time series of observations recorded at a monitoring site. You can create this metadata using the web user interface of the Data Management App, or you can do it using code, which we are demonstrating here.

HydroServer uses a modified version of the OGC SensorThings API data model for storing time series data and their associated metadata. HydroServer's data model includes the following important entities that we need to create before loading data:

* **Sensor**: The instrument or method used to measure or create the Observation values.
* **Observed Property**: The variable that is measured (e.g., discharge, water temperature, etc.).
* **Units of Measure**: The units of measure associated with the Observation values (e.g, cubic meters per second).
* **Processing Level**: The degree of processing that has been applied to the Observation values (e.g., "Raw" or "Quality Controlled").
* **Datastream**: A description of the time series that includes all of these attributes.

Once all of these metadata tables have been populated, the time series of data values can be loaded to the **Observations** table in the database.

**NOTE**: To create objects in HydroServer, you will have to pass their required and optional metadata elements. For more information about HydroServer's data model and a data dictionary that describes each of the entities, see HydroServer's documentation at https://www.hydroserver.org.

### **Create a Sensor**

The OGC SensorThings API data model refers to the method used for creating observations as the "Sensor". In many cases this will be a physical sensor installed at the monitoring site. But, sometimes other methods are used to create observations. We need to create the metadata describing this so potential data users know how the data were created.

**NOTE**: The specific metadata required when creating metadata for a Sensor is dependent upon the "Method Type". For instrument deployments, specific information about the manufacturer and model of the sensor should be specified. For "Derivation" methods, the name and description are required, and a method_code and method_link can be specified if needed.

In [ ]:
real_time_stage_sensor = hs.sensors.create(
    workspace=workspace_id,
    name='Kanzenze Real time Stage Observations',
    description='Real time stage observations recorded at the Kanzenze hydrological station.',
    encoding_type='application/json',
    method_type='Observation',
    method_code='kanzenze-real-time-stage'
)

print("Created observed property:")
print(f"{real_time_stage_sensor.name}: {real_time_stage_sensor.uid}")

### **Create an Observed Property**

Observed Properties are the variables measured at a monitoring site. Similar to creating a monitoring site (Thing), we need to define the required and optional metadata elements for the Observed Property.


In [ ]:
stage = hs.observedproperties.create(
    workspace=workspace_id,
    name='Stage',
    definition='Stage',
    description='Stage is the height of the water surface at a monitoring location relative to a reference level.',
    observed_property_type='Hydrology',
    code='Stage'
)

print("Created observed property:")
print(f"{stage.name}: {stage.uid}")

Created observed property:
Stage: 019ffb16-2e1d-76a5-85ae-38a8d7324054


### **Create Units of Measure**

Next we need to add metadata to specify the Units of measure used for recording the data in the CSV file.

In [ ]:
stage_unit = hs.units.create(
    workspace=workspace_id,
    name='Meter',
    symbol='m',
    definition='Unit for water stage',
    unit_type='Length'
)

print("Created unit:")
print(f"{stage_unit.name}: {stage_unit.uid}")

Created unit:
Meter: 019ffb16-3420-73ae-a68e-4b0bcc148501


### **Create a Processing Level**

In HydroServer, the Processing Level indicates the degree of processing a datastream has been subject to. For example, data can be "Raw", which means that they were recorded in the field and nobody has looked at them yet, or they could be "Quality Controlled", which means that a technician has reviewed the data. All of the data we are loading right now are raw observations from the field with no processing, so we need a Processing Level that indicates this.

In [ ]:
new_processing_level = hs.processinglevels.create(
    workspace=workspace_id,
    code='Raw',
    definition='Raw Data',
    explanation='Data that have not been processed or quality controlled.'
)

print("Created processing levels:")
print(f"{new_processing_level.code}: {new_processing_level.uid}")

Created processing levels:
Raw: 019ffb16-397d-73ac-ab57-1c80ad1616fb


### **Configure the Datastream**

The last step before loading data is to create metadata for each of the "Datastreams". This helps us link the time series values to where they were measured, which Observed Property they represent, which Units they are recorded in, etc. In the following code, we create the necessary datastream metadata, using the UUIDs for the other metadata entities we created above for one datastream we want to load data to.

**NOTE**: Since theDatastream is new, it does not contain any Observation values yet. We'll set the ```value_count=0``` and set the ```phenomenon_begin_time``` to the date the observations started. Those will get reset when we load Observation values. The Datastream has a name and description that we'll set using some attributes of the Datastream, but you can name it according to your own naming conventions.

In [ ]:
ds_stage = hs.datastreams.create(
    name=f"{stage.name} - Real-time - {new_thing.name}",
    description=f'Real-time {stage.name.lower()} observations at {new_thing.name}',
    thing=new_thing.uid,
    sensor=real_time_stage_sensor.uid,
    observed_property=stage.uid,
    processing_level=new_processing_level.uid,
    unit=stage_unit.uid,
    observation_type='Field Observation',
    result_type='Timeseries',
    sampled_medium='Surface Water',
    no_data_value=-9999,
    aggregation_statistic='Continuous',
    time_aggregation_interval=0,
    time_aggregation_interval_unit='minutes',
    intended_time_spacing=1,
    intended_time_spacing_unit='days',
    status='Complete',
    value_count=0,
    phenomenon_begin_time=datetime(year=2022, month=9, day=28),
    is_private=False,
    is_visible=True
)

print("Created datastream:")
print(f"{ds_stage.name}: {ds_stage.uid}")

Created datastream:
Stage - Real-time - Kanzenze Hydrological Station: 019ffb1a-0b79-72b4-967b-27e2af807354
